In [3]:
import pandas as pd

# Load campaign performance data
df_ads = pd.read_csv(r'C:\Users\User\OneDrive\Documents\meta_ads_raw.csv')

# Load CRM customer data
df_crm = pd.read_csv(r'C:\Users\User\OneDrive\Documents\crm_customers.csv')

print("Ads shape:", df_ads.shape)
print("CRM shape:", df_crm.shape)

df_crm.head()

Ads shape: (16, 8)
CRM shape: (15, 9)


,customer_id,campaign_name,channel,age,gender,location,customer_type,purchase_value,purchase_date
0,C001,Summer Sale - Video,Meta,28,Female,Nairobi,New,4200,7/1/2024
1,C002,Retargeting - Dynamic,Meta,35,Male,Mombasa,Returning,8900,7/1/2024
2,C003,Summer Sale - Carousel,Meta,24,Female,Kisumu,New,2400,7/1/2024
3,C004,Brand Awareness - Video,Meta,42,Male,Nairobi,New,350,7/1/2024
4,C005,Retargeting - Dynamic,Meta,31,Female,Nakuru,Returning,9200,14/01/2024


In [ ]:
# Merge on campaign_name — connecting ad performance to customer behaviour
df_merged = pd.merge(df_ads, df_crm, on='campaign_name', how='left')

print("Merged shape:", df_merged.shape)
df_merged.head()

In [5]:
print("Columns after merge:")
print(df_merged.columns.tolist())

print("\nNulls after merge:")
print(df_merged.isnull().sum())

Columns after merge:
['campaign_name', 'channel_x', 'impressions', 'clicks', 'spend', 'conversions', 'revenue', 'date', 'customer_id', 'channel_y', 'age', 'gender', 'location', 'customer_type', 'purchase_value', 'purchase_date']

Nulls after merge:
campaign_name     0
channel_x         0
impressions       3
clicks            0
spend             0
conversions       3
revenue           0
date              0
customer_id       0
channel_y         0
age               0
gender            0
location          0
customer_type     0
purchase_value    0
purchase_date     0
dtype: int64


In [6]:
# Drop channel_y — we only need one channel column
df_merged = df_merged.drop(columns=['channel_y'])

# Rename channel_x back to channel
df_merged = df_merged.rename(columns={'channel_x': 'channel'})

print("Columns now:")
print(df_merged.columns.tolist())

Columns now:
['campaign_name', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'revenue', 'date', 'customer_id', 'age', 'gender', 'location', 'customer_type', 'purchase_value', 'purchase_date']


In [7]:
# Which campaign attracts the highest value customers?
campaign_value = df_merged.groupby('campaign_name').agg(
    avg_purchase_value=('purchase_value', 'mean'),
    total_purchase_value=('purchase_value', 'sum'),
    customer_count=('customer_id', 'count')
).round(2)

campaign_value.sort_values('avg_purchase_value', ascending=False)

,avg_purchase_value,total_purchase_value,customer_count
campaign_name,,,
Retargeting - Dynamic,9300.00,186000,20
Summer Sale - Video,4600.00,73600,16
Summer Sale - Carousel,2566.67,30800,12
Brand Awareness - Video,350.00,4200,12


In [8]:
# New vs Returning customers — who spends more?
customer_type = df_merged.groupby(['campaign_name', 'customer_type']).agg(
    avg_purchase=('purchase_value', 'mean'),
    count=('customer_id', 'count')
).round(2)

customer_type

avg_purchase  count
campaign_name           customer_type                     
Brand Awareness - Video New                   350.0     12
Retargeting - Dynamic   Returning            9300.0     20
Summer Sale - Carousel  New                  2500.0      8
                        Returning            2700.0      4
Summer Sale - Video     New                  4600.0     16

In [9]:
# New vs Returning customers — who spends more?
customer_type = df_merged.groupby(['campaign_name', 'customer_type']).agg(
    avg_purchase=('purchase_value', 'mean'),
    count=('customer_id', 'count')
).round(2)

customer_type


avg_purchase  count
campaign_name           customer_type                     
Brand Awareness - Video New                   350.0     12
Retargeting - Dynamic   Returning            9300.0     20
Summer Sale - Carousel  New                  2500.0      8
                        Returning            2700.0      4
Summer Sale - Video     New                  4600.0     16

In [10]:
# Which city drives the most value?
location_analysis = df_merged.groupby('location').agg(
    total_purchase=('purchase_value', 'sum'),
    avg_purchase=('purchase_value', 'mean'),
    customer_count=('customer_id', 'count')
).round(2)

location_analysis.sort_values('total_purchase', ascending=False)

,total_purchase,avg_purchase,customer_count
location,,,
Nairobi,129800,5408.33,24
Mombasa,66000,5500.00,12
Kisumu,59200,4933.33,12
Nakuru,39600,3300.00,12


In [11]:
output_path = r'C:\Users\User\OneDrive\Documents\marketing-analytics-python\outputs\wrangling_report.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_merged.to_excel(writer, sheet_name='Merged Data', index=False)
    campaign_value.to_excel(writer, sheet_name='Campaign Value')
    customer_type.to_excel(writer, sheet_name='Customer Type')
    location_analysis.to_excel(writer, sheet_name='Location Analysis')

print("Module 2 report saved successfully.")

Module 2 report saved successfully.
